# Data pipeline

Läser in players.csv, clubs.csv och events.csv, städar och räknar ut det som behövs, och exporterar allt som JSON till frontend.

In [82]:
import pandas as pd
import numpy as np
import os

os.makedirs("../frontend/src/data", exist_ok=True)

## 1. Spelare

Alla värderade spelare, senaste säsongen, en rad per spelare.

Jag delar upp kolumnerna i två grupper: de jag verkligen behöver (id, namn, säsong, kontrakt, värde) och de som bara är trevliga att ha (längd, vikt, fot). Saknas något av det viktiga tar jag bort raden, annars går det inte att räkna ut kontraktstid eller visa ett värde. Saknas bara längd eller vikt spelar det mindre roll, då får de bara stå tomma.

En del spelare dyker upp två gånger samma säsong. Vet inte helt säkert varför, gissade först på klubbyte men kan inte bevisa det eftersom den här filen inte har någon klubbkolumn. Jag behåller bara den senaste värderingen och struntar i varför.

Det finns data för två säsonger, men jag vill bara ha den senaste, så varje spelare bara syns en gång.

In [83]:
relevant_columns = [
    "playerId",
    "playerName",
    "seasonName",
    "contractExpiration",
    "snapshot_date",
    "fairPrice",
    "playerImageUrl",
    "foot",
    "dateOfBirth",
    "height",
    "weight",
    "passportAreaName",
    "birthPlaceAreaName",
]

critical_columns = [
    "playerId",
    "playerName",
    "seasonName",
    "contractExpiration",
    "snapshot_date",
    "fairPrice",
]

players = pd.read_csv("../data/players.csv")
players_cleaned = players[relevant_columns].dropna(subset=critical_columns).copy()

players_cleaned["contractExpiration"] = pd.to_datetime(
    players_cleaned["contractExpiration"]
)
players_cleaned["snapshot_date"] = pd.to_datetime(players_cleaned["snapshot_date"])

players_valued = players_cleaned.sort_values("snapshot_date").drop_duplicates(
    subset=["playerId", "seasonName"], keep="last"
)

players_valued["years_left_on_contract"] = (
    (players_valued["contractExpiration"] - players_valued["snapshot_date"]).dt.days
    / 365.25
).round(1)

latest_season = players_valued["seasonName"].max()
players_valued_current = players_valued[players_valued["seasonName"] == latest_season]

print(
    f"{len(players_valued_current)} värderade spelare, senaste säsongen ({latest_season})"
)

492 värderade spelare, senaste säsongen (2024/2025)


## 2. Kolla värdet från förra säsongen

Vill kunna visa om en spelares värde gått upp eller ner sen förra säsongen.

In [84]:
events_columns = ["playerId", "clubId", "clubName", "seasonName"]
events = pd.read_csv("../data/events.csv", usecols=events_columns)

player_club_counts = (
    events.groupby(["playerId", "seasonName", "clubId", "clubName"])
    .size()
    .reset_index(name="event_count")
)

player_club_mapping = player_club_counts.sort_values(
    "event_count", ascending=False
).drop_duplicates(subset=["playerId", "seasonName"])

print(f"{len(player_club_mapping)} spelare och klubb kopplingar")

1126 spelare och klubb kopplingar


In [85]:
players_with_club = players_valued_current.merge(
    player_club_mapping[["playerId", "seasonName", "clubId", "clubName"]],
    on=["playerId", "seasonName"],
    how="left",
)
print(
    f"Utan klubbmatchning: {players_with_club['clubName'].isna().sum()} av {len(players_with_club)}"
)

Utan klubbmatchning: 0 av 492


In [86]:
season_order = sorted(players_valued["seasonName"].unique())
prev_season = season_order[-2] if len(season_order) > 1 else None

if prev_season:
    prev_values = players_valued[players_valued["seasonName"] == prev_season][
        ["playerId", "fairPrice"]
    ].copy()
    prev_values["fairPricePrevM"] = (prev_values["fairPrice"] / 1e6).round(1)
    prev_values = prev_values[["playerId", "fairPricePrevM"]]
else:
    prev_values = pd.DataFrame(columns=["playerId", "fairPricePrevM"])

## 3. Alla spelare

En lista med alla värderade spelare och vilken klubb de tillhör. Används både för att visa en klubbs trupp och för att söka fram en spelare inom den truppen.

In [87]:
all_players = players_with_club.merge(prev_values, on="playerId", how="left")
all_players["fairPriceM"] = (all_players["fairPrice"] / 1e6).round(1)
all_players["yearsLeftOnContract"] = all_players["years_left_on_contract"]
all_players["contractExpiration"] = all_players["contractExpiration"].dt.strftime(
    "%Y-%m-%d"
)

all_players_export = all_players[
    [
        "playerId",
        "playerName",
        "clubName",
        "fairPriceM",
        "fairPricePrevM",
        "yearsLeftOnContract",
        "contractExpiration",
        "playerImageUrl",
        "foot",
        "dateOfBirth",
        "height",
        "weight",
        "passportAreaName",
        "birthPlaceAreaName",
    ]
]
all_players_export.to_json(
    "../frontend/src/data/all_players_data.json", orient="records", indent=2
)
print(f"all_players_data.json: {len(all_players_export)} players")

all_players_data.json: 492 players


## 4. Matcher

Alla spelade matcher, med den senaste matchen först.

In [88]:
match_columns = [
    "competitionId",
    "seasonName",
    "matchId",
    "label",
    "status",
    "dateutc",
    "homeClubId",
    "homeClubName",
    "homeClubImageurl",
    "homeFullTimeScore",
    "awayClubId",
    "awayClubName",
    "awayClubImageurl",
    "awayFullTimeScore",
]

events_matches = pd.read_csv("../data/events.csv", usecols=match_columns)
matches = events_matches.drop_duplicates(subset=["matchId"]).sort_values(
    "dateutc", ascending=False
)

matches_export = matches[
    [
        "matchId",
        "seasonName",
        "dateutc",
        "homeClubId",
        "homeClubName",
        "homeClubImageurl",
        "homeFullTimeScore",
        "awayClubId",
        "awayClubName",
        "awayClubImageurl",
        "awayFullTimeScore",
    ]
]
matches_export.to_json(
    "../frontend/src/data/matches_data.json", orient="records", indent=2
)
print(f"matches_data.json: {len(matches_export)} matcher")

matches_data.json: 760 matcher


## 5. Spelarstatistik

Mål, assist och xG per spelare, denna säsong och förra säsongen, plus vanligaste position.

In [89]:
extended_columns = [
    "matchId",
    "seasonName",
    "playerId",
    "playerName",
    "playerPosition",
    "clubId",
    "clubName",
    "GOALS",
    "ASSISTS",
    "SHOT_XG",
]
events_extended = pd.read_csv("../data/events.csv", usecols=extended_columns)

player_stats_all_seasons = (
    events_extended.groupby(["playerId", "seasonName"])[["GOALS", "ASSISTS", "SHOT_XG"]]
    .sum()
    .reset_index()
)

current_stats = player_stats_all_seasons[
    player_stats_all_seasons["seasonName"] == latest_season
].rename(
    columns={
        "GOALS": "goalsCurrent",
        "ASSISTS": "assistsCurrent",
        "SHOT_XG": "xgCurrent",
    }
)

prev_stats = player_stats_all_seasons[
    player_stats_all_seasons["seasonName"] != latest_season
].rename(columns={"GOALS": "goalsPrev", "ASSISTS": "assistsPrev", "SHOT_XG": "xgPrev"})

def most_common_position(positions):
    mode_result = positions.mode()
    if mode_result.empty:
        return None
    return mode_result.iloc[0]

player_positions = (
    events_extended.groupby("playerId")["playerPosition"]
    .agg(most_common_position)
    .reset_index()
)

player_stats = current_stats.merge(
    prev_stats[["playerId", "goalsPrev", "assistsPrev", "xgPrev"]],
    on="playerId",
    how="left",
).merge(player_positions, on="playerId", how="left")
player_stats["xgCurrent"] = player_stats["xgCurrent"].round(2)
player_stats["xgPrev"] = player_stats["xgPrev"].round(2)
player_stats = player_stats.drop(columns=["seasonName"])

player_stats.to_json(
    "../frontend/src/data/player_stats_data.json", orient="records", indent=2
)
print(f"player_stats_data.json: {len(player_stats)} spelare")

player_stats_data.json: 559 spelare


## 6. Klubbform

Vinster, oavgjorda och förluster per klubb, bara den senaste säsongen.

In [90]:
matches_current_season = matches[matches["seasonName"] == latest_season]


def match_result(row, opp_score_col, own_score_col):
    if row[own_score_col] > row[opp_score_col]:
        return "win"
    elif row[own_score_col] < row[opp_score_col]:
        return "loss"
    return "draw"


home = matches_current_season.copy()
home["clubId"] = home["homeClubId"]
home["result"] = home.apply(
    lambda row: match_result(row, "awayFullTimeScore", "homeFullTimeScore"), axis=1
)
home["goalsFor"] = home["homeFullTimeScore"]
home["goalsAgainst"] = home["awayFullTimeScore"]

away = matches_current_season.copy()
away["clubId"] = away["awayClubId"]
away["result"] = away.apply(
    lambda row: match_result(row, "homeFullTimeScore", "awayFullTimeScore"), axis=1
)
away["goalsFor"] = away["awayFullTimeScore"]
away["goalsAgainst"] = away["homeFullTimeScore"]

all_results = pd.concat([home, away])

wins = (
    all_results[all_results["result"] == "win"]
    .groupby("clubId")
    .size()
    .reset_index(name="win")
)
draws = (
    all_results[all_results["result"] == "draw"]
    .groupby("clubId")
    .size()
    .reset_index(name="draw")
)
losses = (
    all_results[all_results["result"] == "loss"]
    .groupby("clubId")
    .size()
    .reset_index(name="loss")
)
goals = all_results.groupby("clubId")[["goalsFor", "goalsAgainst"]].sum().reset_index()

club_form = (
    wins.merge(draws, on="clubId", how="outer")
    .merge(losses, on="clubId", how="outer")
    .merge(goals, on="clubId", how="outer")
    .fillna(0)
)

club_form["points"] = club_form["win"] * 3 + club_form["draw"]
club_form["goalDifference"] = club_form["goalsFor"] - club_form["goalsAgainst"]

club_form = club_form.sort_values(
    ["points", "goalDifference"], ascending=False
).reset_index(drop=True)
club_form["rank"] = club_form.index + 1


print(f"Klubbform beräknad för {len(club_form)} klubbar")

Klubbform beräknad för 20 klubbar


## 7. Klubbsammanfattning

En rad per klubb: truppstorlek, truppvärde, och formen från förra sektionen. Det här blir huvudkortet för klubbsidan.

In [91]:
club_columns = [
    "clubId",
    "clubName",
    "clubimageurl",
    "seasonName",
    "transferkpi",
    "totalRevenues",
    "netProfit",
]
clubs = pd.read_csv("../data/clubs.csv", usecols=club_columns)
clubs_current = clubs[clubs["seasonName"] == latest_season]

squad_summary = (
    players_with_club.groupby("clubId")
    .agg(
        numPlayers=("playerName", "count"),
        avgValueM=("fairPrice", lambda x: round(x.mean() / 1e6, 1)),
        totalValueM=("fairPrice", lambda x: round(x.sum() / 1e6, 1)),
    )
    .reset_index()
)

club_summary = clubs_current.merge(squad_summary, on="clubId", how="left").merge(
    club_form, on="clubId", how="left"
)

club_summary_export = club_summary[
    [
        "clubId",
        "clubName",
        "clubimageurl",
        "transferkpi",
        "totalRevenues",
        "netProfit",
        "numPlayers",
        "avgValueM",
        "totalValueM",
        "win",
        "draw",
        "loss",
        "points",
        "goalsFor",
        "goalsAgainst",
        "goalDifference",
        "rank",
    ]
]

club_summary_export.to_json(
    "../frontend/src/data/club_summary_data.json", orient="records", indent=2
)
print(f"club_summary_data.json: {len(club_summary_export)} klubbar")

club_summary_data.json: 20 klubbar
